In [ ]:
# Thu thập tất cả Laptop tại Phong Vũ
import pandas as pd
import time
import re
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def extract_laptop_info(card):
    try:
        title_elem = card.select_one("div.att-product-card-title h3")
        if not title_elem: return None
            
        ten_raw = title_elem.text.strip()
        ten = ten_raw.split('(')[0].strip()
        
        a_tag = card.select_one("a")
        url_sp = a_tag["href"] if a_tag and "href" in a_tag.attrs else ""
        if url_sp and not url_sp.startswith("http"):
            url_sp = "https://phongvu.vn" + url_sp
            
        try:
            gia_ht_elem = card.select_one("div.att-product-detail-latest-price")
            gia_hien_tai = int(re.sub(r"[^\d]", "", gia_ht_elem.text)) if gia_ht_elem else 0
        except:
            gia_hien_tai = 0

        try:
            gia_goc_elem = card.select_one("del") or card.select_one(".retail-price") or card.select_one("div.att-product-detail-retail-price")
            gia_goc = int(re.sub(r"[^\d]", "", gia_goc_elem.text)) if gia_goc_elem else gia_hien_tai
        except:
            gia_goc = gia_hien_tai 

        if gia_goc < gia_hien_tai:
            gia_goc = gia_hien_tai
            
        muc_giam = gia_goc - gia_hien_tai
        phan_tram_giam = f"{round((muc_giam / gia_goc * 100), 1):g}%" if gia_goc > 0 and muc_giam > 0 else "0%"

        card_text = card.get_text(separator=" ", strip=True)

        ram_match = re.search(r'\b(4|8|16|24|32|64)\s*GB\b', card_text, re.IGNORECASE)
        ram = ram_match.group(0).upper() if ram_match else "N/A"

        ssd_match = re.search(r'\b(128|256|512)\s*GB\b|\b(1|2)\s*TB\b', card_text, re.IGNORECASE)
        ssd = ssd_match.group(0).upper() if ssd_match else "N/A"

        cpu_pattern = r'\b(i[3579]-?\w+|U[579]-?\w+|R[3579]\s?\w+|Core\s?[357]\s?\w+|Apple\s?M\d\w*)\b'
        cpu_match = re.search(cpu_pattern, card_text, re.IGNORECASE)
        cpu = cpu_match.group(0) if cpu_match else "N/A"

        gpu_pattern = r'\b(RTX\s?\d{4}|GTX\s?\d{4}|Radeon\w*|Intel\s?Graphics|Arc\s?Graphics|Apple\s?GPU)\b'
        gpu_match = re.search(gpu_pattern, card_text, re.IGNORECASE)
        gpu = gpu_match.group(0) if gpu_match else "Onboard"

        if ten and gia_hien_tai > 0:
            return {
                "Tên sản phẩm": ten, "CPU": cpu, "RAM": ram, "SSD": ssd, "VGA/GPU": gpu,
                "Giá gốc": gia_goc, "Giá hiện tại": gia_hien_tai, 
                "Mức giảm": muc_giam, "% Giảm giá": phan_tram_giam, "URL": url_sp
            }
        return None
    except Exception:
        return None

def scrape_phong_vu_data(so_luong_can_lay, url_template, mo_ta_chuyen_muc):
    print(f"--- Đang bắt đầu thu thập dữ liệu: {mo_ta_chuyen_muc} ---")
    results = []
    page = 1

    while len(results) < so_luong_can_lay:
        url = url_template.format(page=page)
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
            if response.status_code != 200: break
                
            soup = BeautifulSoup(response.content, "html.parser")
            cards = soup.select("div.product-card") or soup.select(".css-137u7ef")
            
            if not cards:
                print(f"Hết sản phẩm ở trang {page}.")
                break

            for card in cards:
                if len(results) >= so_luong_can_lay: break
                info = extract_laptop_info(card)
                if info:
                    results.append(info)
                    print(f"[{len(results)}] {info['Tên sản phẩm']} | {info['Giá hiện tại']}đ (-{info['% Giảm giá']})")
            
            page += 1
            time.sleep(1.5)
            
        except Exception as e:
            print(f"Lỗi: {e}")
            break

    return pd.DataFrame(results)

if __name__ == "__main__":
    SO_LUONG = 200
    URL_LAPTOP = "https://phongvu.vn/c/laptop?page={page}"

    df_laptop = scrape_phong_vu_data(SO_LUONG, URL_LAPTOP, "Tất cả Laptop Phong Vũ")
    
    if not df_laptop.empty:
        df_laptop.to_csv("Laptop_PhongVu.csv", index=False, encoding='utf-8-sig')
        print(f"\nĐã lưu {len(df_laptop)} sản phẩm vào file 'Laptop_PhongVu.csv'")

--- Đang bắt đầu thu thập dữ liệu: Tất cả Laptop Phong Vũ ---
[1] Laptop Msi Katana B14WEK-286VN | 33190000đ (-21%)
[2] Laptop Asus Vivobook S14 S3407CA-LY068WS | 23490000đ (-16.1%)
[3] Laptop Lenovo IdeaPad Slim 3 15ARP10 - 83K700GDVN | 17990000đ (-9.6%)
[4] Laptop HP 15-fc0024AU - D0BH2PA | 16490000đ (-6.3%)
[5] Laptop HP Victus 15 fa2731TX | 28190000đ (-6%)
[6] Laptop Asus Vivobook 14 X1404VA-EB509W | 13490000đ (-20.6%)
[7] Laptop Acer Aspire 7 A715-59G-57TU | 21490000đ (-2.3%)
[8] Laptop HP 15-fd0823TU - C81NMPA | 21990000đ (-8.3%)
[9] Laptop Msi Modern 15 F13MG-667VN_16G | 16490000đ (-13.2%)
[10] Laptop HP Victus 15-fb3116AX - BX8U4PA | 28690000đ (-2.7%)
[11] Laptop Asus TUF Gaming F16 FX607VU-RL045W | 28490000đ (-1.7%)
[12] Laptop Acer Aspire Lite 15 AL15-72P-581V | 17590000đ (-7.4%)
[13] Laptop Asus Vivobook 15 X1502VA-BQ886W | 20990000đ (-4.5%)
[14] Laptop HP 250 G10 - A06FFPT | 23590000đ (-2.5%)
[15] Laptop Dell 15 DC15250 - CPH997 | 23490000đ (-0%)
[16] Laptop Msi Cyborg 15 A

In [2]:
# Thu thập Laptop bán chạy nhất tại Phong Vũ
import pandas as pd
import time
import re
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def extract_laptop_info(card):
    try:
        title_elem = card.select_one("div.att-product-card-title h3")
        if not title_elem: return None
            
        ten_raw = title_elem.text.strip()
        ten = ten_raw.split('(')[0].strip()
        
        a_tag = card.select_one("a")
        url_sp = a_tag["href"] if a_tag and "href" in a_tag.attrs else ""
        if url_sp and not url_sp.startswith("http"):
            url_sp = "https://phongvu.vn" + url_sp
            
        try:
            gia_ht_elem = card.select_one("div.att-product-detail-latest-price")
            gia_hien_tai = int(re.sub(r"[^\d]", "", gia_ht_elem.text)) if gia_ht_elem else 0
        except:
            gia_hien_tai = 0

        try:
            gia_goc_elem = card.select_one("del") or card.select_one(".retail-price") or card.select_one("div.att-product-detail-retail-price")
            gia_goc = int(re.sub(r"[^\d]", "", gia_goc_elem.text)) if gia_goc_elem else gia_hien_tai
        except:
            gia_goc = gia_hien_tai 

        if gia_goc < gia_hien_tai:
            gia_goc = gia_hien_tai
            
        muc_giam = gia_goc - gia_hien_tai
        phan_tram_giam = f"{round((muc_giam / gia_goc * 100), 1):g}%" if gia_goc > 0 and muc_giam > 0 else "0%"

        card_text = card.get_text(separator=" ", strip=True)

        ram_match = re.search(r'\b(4|8|16|24|32|64)\s*GB\b', card_text, re.IGNORECASE)
        ram = ram_match.group(0).upper() if ram_match else "N/A"

        ssd_match = re.search(r'\b(128|256|512)\s*GB\b|\b(1|2)\s*TB\b', card_text, re.IGNORECASE)
        ssd = ssd_match.group(0).upper() if ssd_match else "N/A"

        cpu_pattern = r'\b(i[3579]-?\w+|U[579]-?\w+|R[3579]\s?\w+|Core\s?[357]\s?\w+|Apple\s?M\d\w*)\b'
        cpu_match = re.search(cpu_pattern, card_text, re.IGNORECASE)
        cpu = cpu_match.group(0) if cpu_match else "N/A"

        gpu_pattern = r'\b(RTX\s?\d{4}|GTX\s?\d{4}|Radeon\w*|Intel\s?Graphics|Arc\s?Graphics|Apple\s?GPU)\b'
        gpu_match = re.search(gpu_pattern, card_text, re.IGNORECASE)
        gpu = gpu_match.group(0) if gpu_match else "Onboard"

        if ten and gia_hien_tai > 0:
            return {
                "Tên sản phẩm": ten, "CPU": cpu, "RAM": ram, "SSD": ssd, "VGA/GPU": gpu,
                "Giá gốc": gia_goc, "Giá hiện tại": gia_hien_tai, 
                "Mức giảm": muc_giam, "% Giảm giá": phan_tram_giam, "URL": url_sp
            }
        return None
    except Exception:
        return None

def scrape_phong_vu_data(so_luong_can_lay, url_template, mo_ta_chuyen_muc):
    print(f"--- Đang bắt đầu thu thập dữ liệu: {mo_ta_chuyen_muc} ---")
    results = []
    page = 1

    while len(results) < so_luong_can_lay:
        url = url_template.format(page=page)
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
            if response.status_code != 200: break
                
            soup = BeautifulSoup(response.content, "html.parser")
            cards = soup.select("div.product-card") or soup.select(".css-137u7ef")
            
            if not cards:
                print(f"Hết sản phẩm ở trang {page}.")
                break

            for card in cards:
                if len(results) >= so_luong_can_lay: break
                info = extract_laptop_info(card)
                if info:
                    results.append(info)
                    print(f"[{len(results)}] {info['Tên sản phẩm']} | {info['Giá hiện tại']}đ (-{info['% Giảm giá']})")
            
            page += 1
            time.sleep(1.5)
            
        except Exception as e:
            print(f"Lỗi: {e}")
            break

    return pd.DataFrame(results)

if __name__ == "__main__":
    SO_LUONG = 200

    URL_LAPTOP_BAN_CHAY = "https://phongvu.vn/c/laptop?sort=SORT_BY_TOP_SALE_QUANTITY_7_DAYS&order=DESC&page={page}"

    df_laptop = scrape_phong_vu_data(SO_LUONG, URL_LAPTOP_BAN_CHAY, "Laptop Bán Chạy Nhất Phong Vũ")
    
    if not df_laptop.empty:
        df_laptop.to_csv("Laptop_BanChay_PhongVu.csv", index=False, encoding='utf-8-sig')
        print(f"\nĐã lưu {len(df_laptop)} sản phẩm vào file 'Laptop_BanChay_PhongVu.csv'")

--- Đang bắt đầu thu thập dữ liệu: Laptop Bán Chạy Nhất Phong Vũ ---
[1] Laptop Msi Katana B14WEK-286VN | 33190000đ (-21%)
[2] Laptop Asus Vivobook S14 S3407CA-LY068WS | 23490000đ (-16.1%)
[3] Laptop Lenovo IdeaPad Slim 3 15ARP10 - 83K700GDVN | 17990000đ (-9.6%)
[4] Laptop HP 15-fc0024AU - D0BH2PA | 16490000đ (-6.3%)
[5] Laptop HP Victus 15 fa2731TX | 28190000đ (-6%)
[6] Laptop Asus Vivobook 14 X1404VA-EB509W | 13490000đ (-20.6%)
[7] Laptop Acer Aspire 7 A715-59G-57TU | 21490000đ (-2.3%)
[8] Laptop HP 15-fd0823TU - C81NMPA | 21990000đ (-8.3%)
[9] Laptop Msi Modern 15 F13MG-667VN_16G | 16490000đ (-13.2%)
[10] Laptop HP Victus 15-fb3116AX - BX8U4PA | 28690000đ (-2.7%)
[11] Laptop Asus TUF Gaming F16 FX607VU-RL045W | 28490000đ (-1.7%)
[12] Laptop Acer Aspire Lite 15 AL15-72P-581V | 17590000đ (-7.4%)
[13] Laptop Asus Vivobook 15 X1502VA-BQ886W | 20990000đ (-4.5%)
[14] Laptop HP 250 G10 - A06FFPT | 23590000đ (-2.5%)
[15] Laptop Dell 15 DC15250 - CPH997 | 23490000đ (-0%)
[16] Laptop Msi Cybo